In [13]:
# ==================================================
# IMPORTAR LIBRERÍAS
# ==================================================

import pandas as pd
import glob

In [14]:
# ==================================================
# LEER TODOS LOS CSV DE SENTINEL
# ==================================================

sentinel_files = glob.glob("../data/raw/sentinel/*.csv")

sentinel_list = []

for file in sentinel_files:

    df = pd.read_csv(file)

    sentinel_list.append(df)

df_sentinel = pd.concat(
    sentinel_list,
    ignore_index=True
)

In [15]:
# ==================================================
# LEER TODOS LOS CSV DE MODIS
# ==================================================

modis_files = glob.glob("../data/raw/modis/*.csv")

modis_list = []

for file in modis_files:

    df = pd.read_csv(file)

    modis_list.append(df)

df_modis = pd.concat(
    modis_list,
    ignore_index=True
)

In [16]:
# ==================================================
# VERIFICACIÓN
# ==================================================

print("Sentinel:")
print(df_sentinel.shape)

print("\nMODIS:")
print(df_modis.shape)

Sentinel:
(70132, 23)

MODIS:
(71545, 11)


In [17]:
# ==================================================
# HACER MERGE FINAL
# ==================================================

dataset_final = pd.merge(
    df_sentinel,
    df_modis,
    on=["point_id", "fecha"],
    how="inner"
)

In [18]:
# ==================================================
# VERIFICAR
# ==================================================

print(dataset_final.shape)

dataset_final.head()

(69683, 32)


,system:index_x,blue,bsi,fecha,green,latitude_x,longitude_x,mndwi,month_x,nbr,...,.geo_x,system:index_y,latitude_y,longitude_y,lst_day_c,lst_night_c,month_y,occurrence_y,year_y,.geo_y
0,0_0_0_0,0.01515,-0.312553,2020-10-01,0.03050,41.438161,2.099767,-0.542042,10.0,0.674365,...,"{""type"":""MultiPoint"",""coordinates"":[]}",0_0_0_0,41.438161,2.099767,18.125,10.965,10.0,0,2020,"{""type"":""MultiPoint"",""coordinates"":[]}"
1,0_1_0_0,0.13450,0.106304,2020-10-01,0.17700,41.455948,2.191665,-0.317261,10.0,0.018942,...,"{""type"":""MultiPoint"",""coordinates"":[]}",0_1_0_0,41.455948,2.191665,20.825,12.565,10.0,0,2020,"{""type"":""MultiPoint"",""coordinates"":[]}"
2,0_2_0_0,0.09050,0.061652,2020-10-01,0.11915,41.376447,2.102462,-0.350239,10.0,0.054307,...,"{""type"":""MultiPoint"",""coordinates"":[]}",0_2_0_0,41.376447,2.102462,21.685,12.930,10.0,0,2020,"{""type"":""MultiPoint"",""coordinates"":[]}"
3,0_3_0_0,0.07690,0.084935,2020-10-01,0.08650,41.396659,2.139652,-0.121158,10.0,-0.036384,...,"{""type"":""MultiPoint"",""coordinates"":[]}",0_3_0_0,41.396659,2.139652,21.835,14.425,10.0,0,2020,"{""type"":""MultiPoint"",""coordinates"":[]}"
4,0_5_0_0,0.06340,0.061221,2020-10-01,0.09945,41.368093,2.043173,-0.362908,10.0,0.189122,...,"{""type"":""MultiPoint"",""coordinates"":[]}",0_5_0_0,41.368093,2.043173,23.155,11.635,10.0,0,2020,"{""type"":""MultiPoint"",""coordinates"":[]}"


In [19]:
# ============================================================
# AÑADIR ELEVACIÓN
# ============================================================

# Añadimos la elevación asociada a cada punto geográfico.
#
# La elevación no cambia con el tiempo, por lo que basta con unir esta información utilizando el identificador
# único de cada punto (point_id).

df_elevation = pd.read_csv(
    "../data/raw/elevation/barcelona_points_elevation.csv"
)

# Nos quedamos únicamente con las columnas necesarias
df_elevation = df_elevation[
    ['point_id', 'elevation']
]

# Hacemos merge con el dataset principal
dataset_final = dataset_final.merge(
    df_elevation,
    on='point_id',
    how='left'
)

In [20]:
# ============================================================
# VERIFICACIÓN
# ============================================================

print(dataset_final.shape)

dataset_final[['point_id', 'elevation']].head()

(69683, 33)


,point_id,elevation
0,0.0,301
1,1.0,27
2,2.0,85
3,3.0,92
4,4.0,15


In [21]:
# ==================================================
# EXPORTAR DATASET FINAL
# ==================================================

# Guardamos el dataset construido tras el merge
# entre Sentinel y MODIS.

dataset_final.to_csv(
    "../data/processed/dataset_final.csv",
    index=False
)

print("Dataset exportado correctamente.")
print(f"Ruta: ../data/processed/dataset_final.csv")
print(f"Shape: {dataset_final.shape}")

Dataset exportado correctamente.
Ruta: ../data/processed/dataset_final.csv
Shape: (69683, 33)
